In [1]:
import os
import pandas as pd
import numpy as np
from glob import glob

# ====================================
# CONFIGURATION
# ====================================
BASE_DIR = "FARS"
OUTPUT_PATH = "FARS/cleaned_data/fars_person_cleaned.csv"

VARS_OF_INTEREST = [
    "PER_NO", "VEH_NO", "PER_TYP", "AGE", "SEX", "INJ_SEV",
    "DRINKING", "DRUGS", "REST_USE", "AIR_BAG", "SEAT_POS",
    "ST_CASE", "STATE", "EJECTION", "WORK_INJ"
]

# ====================================
# LOAD AND CLEAN EACH YEAR
# ====================================
def load_person_file(year_folder):
    """Load person.csv or Person.csv for a given year folder."""
    candidates = glob(os.path.join(year_folder, "[Pp]erson.csv"))
    if not candidates:
        print(f"No person file found in {year_folder}")
        return None
    file_path = candidates[0]
    try:
        df = pd.read_csv(file_path, encoding="utf-8", low_memory=False)
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding="latin1", low_memory=False)
    df.columns = df.columns.str.upper().str.strip()
    df["YEAR"] = int(os.path.basename(year_folder))
    return df


# ====================================
# COLUMN CLEANING FUNCTIONS
# ====================================
def airbagclean(col): return col.where(~col.isin([0, 97, 98, 99]), pd.NA)
def restuseclean(col): return col.where(~col.isin([0, 20, 29, 96, 98, 99]), pd.NA)
def sexclean(col): return col.where(~col.isin([8, 9]), pd.NA)
def injsevclean(col): return col.where(~col.isin([9]), pd.NA)
def drinkclean(col): return col.where(~col.isin([8, 9]), pd.NA)
def ageclean(col):
    col = col.where(~col.isin([998, 999]), pd.NA)
    col = col.where(col <= 120, pd.NA)
    return col
def drugclean(col): return col.where(~col.isin([8, 9]), pd.NA)
def pertypeclean(col): return col.where(~col.isin([9, 19, 88, 99]), pd.NA)
def ejectionclean(col): return col.where(~col.isin([7, 8, 9]), pd.NA)
def seatposclean(col): return col.where(~col.isin([0, 56, 98, 99]), pd.NA)
def workinjclean(col): return col.where(~col.isin([8, 9]), pd.NA)


# ====================================
# APPLY CLEANING
# ====================================
def clean_person_data(df):
    """Apply harmonized column cleaning."""
    df = df.copy()

    for col, func in {
        "SEX": sexclean,
        "INJ_SEV": injsevclean,
        "DRINKING": drinkclean,
        "AIR_BAG": airbagclean,
        "AGE": ageclean,
        "DRUGS": drugclean,
        "REST_USE": restuseclean,
        "PER_TYP": pertypeclean,
        "WORK_INJ": workinjclean,
        "EJECTION": ejectionclean,
        "SEAT_POS": seatposclean,
    }.items():
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            df[col] = func(df[col])

    return df


# ====================================
# COMBINE ALL YEARS
# ====================================
def combine_person_data(base_dir):
    """Combine all yearly person files into one harmonized dataset."""
    year_folders = sorted([f.path for f in os.scandir(base_dir) if f.is_dir()])
    all_dfs = []

    for folder in year_folders:
        df = load_person_file(folder)
        if df is None:
            continue

        cols = [c for c in VARS_OF_INTEREST if c in df.columns]
        df = df[cols + ["YEAR"]]
        df = clean_person_data(df)

        # Create CASE_ID
        df["ID"] = "FARS_" + df["YEAR"].astype(str) + "_" + df["ST_CASE"].astype(str)
        all_dfs.append(df)

        print(f"Processed {os.path.basename(folder)}: {df.shape[0]} rows")

    return pd.concat(all_dfs, ignore_index=True)


# ====================================
# DATA QUALITY CHECK
# ====================================
def summarize_data_quality(df):
    print("\n=== DATA QUALITY SUMMARY ===")
    missing_pct = df.isna().mean() * 100
    print("Missingness (%):")
    print(missing_pct.sort_values(ascending=False))
    print("\nRanges / Unique Values:")
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            print(f"{col}: min={df[col].min()}, max={df[col].max()}")
        else:
            print(f"{col}: {df[col].nunique()} unique values")


# ====================================
# MAIN SCRIPT
# ====================================
if __name__ == "__main__":
    os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

    fars_person = combine_person_data(BASE_DIR)
    summarize_data_quality(fars_person)

    # Duplicate check
    dupes = fars_person.duplicated(subset=["ID", "VEH_NO", "PER_NO"], keep=False)
    dup_df = fars_person.loc[dupes, ["ID", "VEH_NO", "PER_NO"]]
    n_dupes = dup_df.shape[0]
    print(f"\nDuplicate check: Found {n_dupes} duplicate person-vehicle-case entries")
    if n_dupes > 0:
        print(dup_df.head())

    fars_person.to_csv(OUTPUT_PATH, index=False)
    print(f"\nCleaned data saved to: {OUTPUT_PATH}")


Processed 2016: 86474 rows
Processed 2017: 85840 rows
Processed 2018: 84344 rows
Processed 2019: 82843 rows
Processed 2020: 86396 rows
Processed 2021: 97511 rows
Processed 2022: 96186 rows
Processed 2023: 92400 rows
No person file found in FARS\cleaned_data

=== DATA QUALITY SUMMARY ===
Missingness (%):
WORK_INJ    58.904569
DRUGS       56.712978
DRINKING    49.251679
REST_USE    40.991497
EJECTION    17.703099
AIR_BAG     17.688070
SEAT_POS    10.686747
AGE          2.534853
SEX          2.074034
INJ_SEV      1.545800
PER_TYP      0.116293
YEAR         0.000000
PER_NO       0.000000
STATE        0.000000
ST_CASE      0.000000
VEH_NO       0.000000
ID           0.000000
dtype: float64

Ranges / Unique Values:
PER_NO: min=1, max=73
VEH_NO: min=0, max=130
PER_TYP: min=1.0, max=13.0
AGE: min=0.0, max=119.0
SEX: min=1.0, max=2.0
INJ_SEV: min=0.0, max=6.0
DRINKING: min=0.0, max=1.0
DRUGS: min=0.0, max=1.0
REST_USE: min=1.0, max=97.0
AIR_BAG: min=1.0, max=28.0
SEAT_POS: min=11.0, max=55.0
ST